In [1]:
import os
import numpy as np
import cv2
import pickle
import supervision as sv
from supervision.annotators.utils import ColorLookup

In [2]:
info_path = '../../data/kitti/kitti_data_info.pkl'
save_dir = '../../data/kitti/img_results'
with open(info_path, 'rb') as f:
    data_info = pickle.load(f)
classes = {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

scene = list(data_info['kitti_raw'].keys())[0]
frame_names = list(data_info['kitti_raw'][scene].keys())
frame_names.sort()
scene_save_dir = f"{save_dir}/{scene}"

fps = 5
w, h = 1242, 375
video_writer = cv2.VideoWriter(f"{scene}.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

thickness = 1
text_scale = 0.5

In [3]:
for name in frame_names:
    img_path = '../.' + data_info['kitti_raw'][scene][name]['img_path']
    img = cv2.imread(img_path)
    save_path = os.path.join(scene_save_dir, name + '.pkl')
    img_result = pickle.load(open(save_path, 'rb'))

    if len(img_result['bboxes']) == 0:
        video_writer.write(img)
        continue

    bboxes = img_result['bboxes']  # xywh format
    labels = img_result['labels']
    scores = img_result['scores']
    ids = img_result['ids']
    masks = img_result['masks']  # list of masks
    xyxy = np.concatenate([bboxes[:, :2] - bboxes[:, 2:] / 2, bboxes[:, :2] + bboxes[:, 2:] / 2], axis=1)

    img_masks = []
    for contour in masks:
        img_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        cv2.drawContours(img_mask, [contour], -1, 255, thickness=cv2.FILLED)
        img_mask = img_mask.astype(bool)
        img_masks.append(img_mask)
    img_masks = np.array(img_masks)
    if ids is not None:
        anno_labels = [classes[label] + f'_{id}' for label, id in zip(labels, ids)]
    else:
        ids = np.arange(len(labels))
        anno_labels = [classes[label] for label in labels]

    detections = sv.Detections(xyxy=xyxy, confidence=scores, class_id=labels, mask=img_masks, tracker_id=ids)
    annotated_image = img.copy()
    annotated_image = sv.MaskAnnotator(
        color_lookup=sv.ColorLookup.TRACK,
        opacity=0.4
    ).annotate(scene=annotated_image, detections=detections)
    annotated_image = sv.BoxAnnotator(
        color_lookup=sv.ColorLookup.TRACK,
        thickness=thickness
    ).annotate(scene=annotated_image, detections=detections)
    annotated_image = sv.LabelAnnotator(
        color_lookup=sv.ColorLookup.TRACK,
        text_scale=text_scale,
        text_padding=0,
        smart_position=True
    ).annotate(scene=annotated_image, detections=detections, labels=anno_labels)

    # cam_infos = scene['samples'][idx]['cams']
    # cam2img = np.eye(4, dtype=np.float32)
    # cam2img[:3, :3] = cam_infos[cam]['cam2img'][:3, :3]
    # lidar2img = cam2img @ cam_infos[cam]['lidar2cam']

    # objects = pseudo_labels[idx]['objects']
    # bbox_3d_masks = []
    # for obj in objects:
    #     if obj['cam'] != cam or obj['bbox_3d'] is None:
    #         continue
    #     x, y, z, l, w, h, yaw = obj['bbox_3d']
    #     bbox_3d_corners = np.array([
    #         [l / 2, w / 2, h / 2],
    #         [-l / 2, w / 2, h / 2],
    #         [-l / 2, -w / 2, h / 2],
    #         [l / 2, -w / 2, h / 2],
    #         [l / 2, w / 2, -h / 2],
    #         [-l / 2, w / 2, -h / 2],
    #         [-l / 2, -w / 2, -h / 2],
    #         [l / 2, -w / 2, -h / 2]
    #     ])
    #     rotation_matrix = np.array([
    #         [np.cos(yaw), -np.sin(yaw), 0],
    #         [np.sin(yaw), np.cos(yaw), 0],
    #         [0, 0, 1]
    #     ])
    #     bbox_3d_corners = (rotation_matrix @ bbox_3d_corners.T).T + np.array([x, y, z + h / 2])
    #     corners_homo = np.hstack((bbox_3d_corners, np.ones((bbox_3d_corners.shape[0], 1))))
    #     corners_img = (lidar2img @ corners_homo.T).T
    #     corners_img = (corners_img[:, :2] / np.maximum(corners_img[:, 2:3], 1e-4)).astype(np.float32)
    #     in_img = (corners_img[:, 0] >= 0) & (corners_img[:, 0] < 1600) & (corners_img[:, 1] >= 0) & (corners_img[:, 1] < 900)
    #     if in_img.sum() == 0:
    #         continue
    #     hull = cv2.convexHull(corners_img).astype(np.int32)
    #     bbox_3d_mask = np.zeros(img.shape[:2], dtype=np.uint8)
    #     cv2.fillPoly(bbox_3d_mask, [hull], 255)
    #     bbox_3d_masks.append(bbox_3d_mask)
    # bbox_3d_masks = np.array(bbox_3d_masks)
    # # 将bbox_3d_masks应用到annotated_image上，通过增加一个混合mask的方式
    # for mask in bbox_3d_masks:
    #     annotated_image[mask == 255] = (0, 255, 0)

    video_writer.write(annotated_image)

video_writer.release()

In [4]:
idx = 35

sample = scene['samples'][idx]
img_path = '.' + sample['cams'][cam]['img_path']
img = cv2.imread(img_path)
save_path = os.path.join(scene_save_dir, cam, os.path.basename(img_path).replace('.jpg', '.pkl'))
img_result = pickle.load(open(save_path, 'rb'))

bboxes = img_result['bboxes']  # xywh format
labels = img_result['labels']
scores = img_result['scores']
ids = img_result['ids']
masks = img_result['masks']  # list of masks
xyxy = np.concatenate([bboxes[:, :2] - bboxes[:, 2:] / 2, bboxes[:, :2] + bboxes[:, 2:] / 2], axis=1)

img_masks = []
for contour in masks:
    img_mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.drawContours(img_mask, [contour], -1, 255, thickness=cv2.FILLED)
    img_mask = img_mask.astype(bool)
    img_masks.append(img_mask)
img_masks = np.array(img_masks)
if ids is not None:
    anno_labels = [classes[label] + f'_{id}' for label, id in zip(labels, ids)]
else:
    ids = np.arange(len(labels))
    anno_labels = [classes[label] for label in labels]

detections = sv.Detections(xyxy=xyxy, confidence=scores, class_id=labels, mask=img_masks, tracker_id=ids)
annotated_image = img.copy()
annotated_image = sv.MaskAnnotator(
    color_lookup=sv.ColorLookup.TRACK,
    opacity=0.4
).annotate(scene=annotated_image, detections=detections)
annotated_image = sv.BoxAnnotator(
    color_lookup=sv.ColorLookup.TRACK,
    thickness=thickness
).annotate(scene=annotated_image, detections=detections)
annotated_image = sv.LabelAnnotator(
    color_lookup=sv.ColorLookup.TRACK,
    text_scale=text_scale,
    text_padding=0,
    smart_position=True
).annotate(scene=annotated_image, detections=detections, labels=anno_labels)

# save
save_path = f"{scene_name}_{cam}_{idx}.jpg"
cv2.imwrite(save_path, annotated_image)

True